In [50]:
import GBM_LCCM
import numpy as np
import pandas as pd
import pylogit as pl
import warnings
from collections import OrderedDict
from sklearn import preprocessing

In [52]:
# load the dataset, which is available in wide format
lccm_data = pd.read_csv('merged_data2.csv')

lccm_data = lccm_data.copy()

In [77]:
lccm_data

,custom_id,Number,record,date,Province,ProvinceOther,MunicipalDist,CensusAgDivision,Gender,GenderOther,...,OQ1,OQ2,OQ3,rev1,rev2,rev3,profit,breakeven,loss,intercept
0,1,1,34,12/06/2024 11:27,3,NaN,Lorne,4.0,1,NaN,...,1,0,0,0,1,0,1,0,0,1.0
1,2,1,34,12/06/2024 11:27,3,NaN,Lorne,4.0,1,NaN,...,0,1,0,0,1,0,1,0,0,1.0
2,3,1,34,12/06/2024 11:27,3,NaN,Lorne,4.0,1,NaN,...,1,0,0,1,0,0,1,0,0,1.0
3,1,2,35,12/06/2024 11:29,2,NaN,RM of Vanscoy,12.0,2,NaN,...,0,0,1,1,0,0,0,1,0,1.0
4,2,2,35,12/06/2024 11:29,2,NaN,RM of Vanscoy,12.0,2,NaN,...,1,0,0,0,1,0,0,1,0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1009,2,338,2711,02/21/2025 17:30,2,NaN,Rm loreburn,NaN,1,NaN,...,0,0,1,1,0,0,1,0,0,1.0
1010,3,338,2711,02/21/2025 17:30,2,NaN,Rm loreburn,NaN,1,NaN,...,0,0,1,0,0,1,1,0,0,1.0
1011,1,339,2771,02/24/2025 05:39,2,NaN,Moose mountain,NaN,1,NaN,...,1,0,0,1,0,0,1,0,0,1.0
1012,2,339,2771,02/24/2025 05:39,2,NaN,Moose mountain,NaN,1,NaN,...,0,1,0,0,1,0,1,0,0,1.0


In [56]:
# set the number of clusters (latent classes) to 2
n_classes = 2

In [58]:
# the class membership model is defined as a Gaussian Bernoulli Mixture Model
# Gaussian Mixture Model is used for continuous variables while Bernoulli Mixture Model for discrete variables

# select the continuous variables for the class membership model
X_Mem_C = lccm_data[['TotalCrop', 'TotalGrass']]
# it is a good practice to standardize the continuous variables for the Gaussian Mixture Model
Standardized_X_Mem_C = preprocessing.scale(X_Mem_C)

# select the discrete variables for the class membership model
X_Mem_D = lccm_data[['Mis','Dar','RN','RL']]
X_Mem_D = X_Mem_D.values

In [60]:
# add a column of ones named'intercept'. This will be used later to define the Alternative-Specific Constants (ASCs).
lccm_data['intercept']=np.ones(lccm_data.shape[0])

In [86]:
# NOTE: Specification and variable names must be in lists of ordered dictionaries.
# class_specific_specs defines the variables names to be used in the specification of the 
# class specific choice model of each class.
# class_specific_labels defines the names associated with each of the variables which
# will be displayed in the output tables after estimation.


class_specific_specs = [OrderedDict([('intercept',[2]),
                                     ('N2',[2]),
                                     ('OQ2',[2]),
                                     ('OQ3',[2]),
                                     ('I1',[2]),
                                     ('I5',[2]),
                                     ('I10',[2]),
                                     ('B25',[2]),
                                     ('B50',[2])]),
                                      
                        OrderedDict([('intercept',[2]),
                                     ('N2',[2]),
                                     ('OQ2',[2]),
                                     ('OQ3',[2]),
                                     ('I1',[2]),
                                     ('I5',[2]),
                                     ('I10',[2]),
                                     ('B25',[2]),
                                     ('B50',[2])])]
                       
class_specific_labels = [OrderedDict([('nq',['N2']),
                                     ('oq2',['OQ2']),
                                     ('oq3',['OQ3']),
                                     ('insur1',['I1']),
                                     ('insur2',['I5']),
                                     ('insur3',['I10']),
                                     ('bio1',['B25']),
                                     ('bio2',['B50'])])]
                         

In [88]:
# Starting Values for the Class-Specific Choice Model paramerters
paramClassSpec = []
paramClassSpec.append(np.array([0,0,0,0,0,0,0,0])) # for the first class
paramClassSpec.append(np.array([0,0,0,0,0,0,0,0])) # for the second class

In [90]:
# Define a location to store the resutls 
outputFilePath = 'ModelResultsLog'

# In this example we will assume that eahc trip was done by different individual
# So, that is why ind_id_col is equal to 'trip_id'


# Train/Estimate the model
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    
    
    GBM_LCCM.lccm_fit(data = lccm_data,
                      X = Standardized_X_Mem_C,
                      X_Dummy = X_Mem_D,
                      dataTest = lccm_data,
                      XTest = Standardized_X_Mem_C,
                      XTest_Dummy = X_Mem_D,
                      prediction_test = 'Yes',
                      GMM_Initialization = 'random',
                      ind_id_col = 'Number', 
                      obs_id_col = 'custom_id',
                      alt_id_col = 'Scenario',
                      choice_col = 'Choice', 
                      n_classes = 2,
                      covariance_type = 'full',
                      reg_covar = 1e-6,
                      tol = 1e-3,
                      max_iter = 100,
                      class_specific_specs = class_specific_specs,
                      class_specific_labels = class_specific_labels,
                      #paramClassSpec = paramClassSpec,
                      outputFilePath = outputFilePath,
                      outputFileName = 'ModelResults')

Processing data


IndexError: index 3 is out of bounds for axis 0 with size 3